In [1]:
import os
from pathlib import Path

import pandas as pd
import polars as pl
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import scipy
import sklearn
import tensorflow as tf
import xgboost as xgb
import lightgbm as lgb
import catboost as cb

import mllabs  # ml-labs: our experiment management framework

import sys
print(sys.version)

for i in [pd, pl, np, plt, sns, scipy, sklearn, tf, xgb, lgb, cb, mllabs]:
    if hasattr(i, '__version__'):
        print(i.__name__, i.__version__)
    else:
        print(i.__name__)

from IPython.display import Markdown

from mllabs import Connector, Experimenter, ColSelector
from mllabs.collector import MetricCollector, ModelAttrCollector, StackingCollector
from mllabs.adapter import XGBoostAdapter, LightGBMAdapter, CatBoostAdapter
from mllabs.nn import NNClassifier
from mllabs.col import ohe_drop_first

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from mllabs.processor import PolarsLoader, PandasConverter, ExprProcessor
from sklearn.pipeline import make_pipeline

data_path = Path('data')

# Binary-encode several categorical features using Polars expressions.
# We also derive indicator columns for internet-service-related features
# ('No internet service' is treated as a separate binary flag).
dict_expr = {
    'gender': (pl.col('gender') == 'Male').cast(pl.Int8),
    'No_Internet': (pl.col('DeviceProtection') == 'No internet service').cast(pl.Int8),
    'DSL_Y': (pl.col('InternetService') == 'DSL').cast(pl.Int8)
}
for i in ['Dependents', 'PaperlessBilling', 'Partner', 'PhoneService']:
    dict_expr[i] = (pl.col(i) == 'Yes').cast(pl.Int8)

for i in ['DeviceProtection', 'OnlineBackup', 'OnlineSecurity', 'StreamingMovies', 'StreamingTV', 'TechSupport', 'MultipleLines']:
    dict_expr[i + '_Y'] = (pl.col(i) == 'Yes').cast(pl.Int8)

# PolarsLoader reads CSVs into Polars DataFrames.
# ExprProcessor applies the expression dict above in a single vectorized pass.
loader = make_pipeline(
    PolarsLoader(predefined_types={'id': pl.Int64}),
    ExprProcessor(dict_expr=dict_expr)
)

df_train = loader.fit_transform([data_path / 'train.csv']).with_columns(
    (pl.col('Churn') == 'Yes').cast(pl.Int8)
)
df_org = loader.transform([data_path / 'WA_Fn-UseC_-Telco-Customer-Churn.csv']).drop('customerID').with_columns(
    id=-pl.int_range(1, pl.len() + 1),
    tenure=pl.col('tenure').clip(1),
    Churn=(pl.col('Churn') == 'Yes').cast(pl.Int8)
)
df_test = loader.transform([data_path / 'test.csv'])

# Variable groupings
# X_bin  : original binary features (Yes/No encoded as 0/1)
# X_tri  : 3-level categorical features (Yes / No / No internet service)
# X_bin2 : derived binary flags from X_tri (e.g. OnlineSecurity_Y)
# X_num  : numerical features
# X_nom  : nominal categoricals kept as strings for tree-based models
X_bin = ['Dependents', 'PaperlessBilling', 'Partner', 'PhoneService', 'gender', 'SeniorCitizen']
X_tri = ['DeviceProtection', 'OnlineBackup', 'OnlineSecurity', 'StreamingMovies', 'StreamingTV', 'TechSupport']
X_bin2 = ['{}_Y'.format(i) for i in X_tri] + ['No_Internet', 'DSL_Y', 'MultipleLines_Y']
X_tri.append('InternetService')
X_tri.append('MultipleLines')
X_num = ['TotalCharges', 'MonthlyCharges', 'tenure']
X_nom = ['PaymentMethod', 'Contract']
X_all = X_bin + X_tri + X_bin2 + X_num + X_nom

target = 'Churn'

2026-03-17 05:16:46.611201: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


3.12.6 (main, Sep 30 2024, 02:19:13) [GCC 9.4.0]
pandas 2.3.3
polars 1.39.0
numpy 2.1.3
matplotlib.pyplot
seaborn 0.13.2
scipy 1.15.2
sklearn 1.5.2
tensorflow 2.20.0
xgboost 2.1.2
lightgbm 4.6.0
catboost 1.2.8
mllabs 0.6.3


In [2]:
if os.path.exists('exp/skf5_2'):
    e_skf5 = Experimenter.load('exp/skf5_2', df_train, aug_data = df_org)
    if e_skf5.status == 'closed':
        e_skf5.reopen_exp()
else:
    e_skf5 = Experimenter.create(
        df_train, 'exp/skf5_2', title='The experimentation using 5-fold StratifiedKFold',
        sp=StratifiedKFold(n_splits=5, shuffle=True, random_state=1),
        splitter_params={'y': target}, aug_data = df_org
    )

e_skf5.set_grp('pre', role='stage', method='transform')
y_edges = {'y': [(None, target)]}
e_skf5.set_grp(
    'clf', role='head', method='predict_proba', edges=y_edges
)
e_skf5.add_collector(
    MetricCollector(
        'AUC',
        Connector(edges=y_edges), slice(-1, None), roc_auc_score, include_train = True
    )
)
e_skf5.add_collector(
    StackingCollector(
        'stacking', Connector(edges=y_edges),
        slice(-1, None), method='mean', experimenter = e_skf5
    )
)

## Neural network
e_skf5.set_grp('nn', parent = 'clf', processor = NNClassifier, params = {'metrics': ['auc'], 'early_stopping': 0, 'validation_fraction': 0})
e_skf5.add_collector(
    ModelAttrCollector('nn_evals', Connector(processor=NNClassifier), result_key='evals_result')
)

📁 Created directory: exp/skf5_2
Collect 5/5 (100%)        
Collect 5/5 (100%)        
Collect 5/5 (100%)        


In [3]:
from sklearn.preprocessing import KBinsDiscretizer, RobustScaler, PolynomialFeatures, StandardScaler, TargetEncoder, MinMaxScaler
from mllabs.processor import FrequencyEncoder, CatConverter, CatPairCombiner, CatOOVFilter, TypeConverter
from sklearn.impute import SimpleImputer
e_skf5.set_node(
    'si', grp='pre', processor=SimpleImputer, edges = {'X': [(None, X_num)]}
)
e_skf5.set_node(
    'kbin_1', grp='pre', processor=KBinsDiscretizer, edges = {'X': [('si', ['.*TotalCharges', '.*MonthlyCharges'])]},
    params = {'n_bins': [4000, 200], 'strategy': 'uniform', 'encode': 'ordinal',  'random_state': 123}
)
e_skf5.set_node(
    'kbin_2', grp='pre', processor=KBinsDiscretizer, edges = {'X': [('si', ['.*TotalCharges', '.*MonthlyCharges'])]},
    params = {'n_bins': [500, 100], 'strategy': 'uniform', 'encode': 'ordinal',  'random_state': 123}
)
e_skf5.set_node(
    'expr1', grp='pre', processor = ExprProcessor, edges = {'X': [('si', None)]},
    params={'dict_expr': {
        'TotalCharges_per_tenure': pl.col('si__TotalCharges') / pl.col('si__tenure'),
        'Total_per_Monthly': pl.col('si__TotalCharges') / pl.col('si__MonthlyCharges'),
        'Monthly_per_Total': pl.col('si__MonthlyCharges') / pl.col('si__TotalCharges'),
        'Monthly_Avg_ratio': pl.col('si__MonthlyCharges') / (pl.col('si__TotalCharges') / pl.col('si__tenure')),
        'tenure_sq': pl.col('si__tenure') ** 2,
    }, 'with_columns': False}
)
e_skf5.set_node(
    'n2s', grp='pre', processor=TypeConverter, method = 'transform', edges = {'X': [('si', None)]}, params={'to': 'str'}
)

e_skf5.set_node(
    'n2c', grp='pre', processor=CatConverter, method = 'transform', edges = {'X': [('n2s', None), ('kbin_1', None), ('kbin_2', None)]}
)

e_skf5.set_node(
    'coov', grp='pre', processor=CatOOVFilter, method = 'transform', edges = {'X': [('n2c', None)]}, params={'min_frequency': 10, 'missing_value': 'OOV'}
)
e_skf5.set_node(
    'com3', grp='pre', processor = CatPairCombiner, edges = {'X': [(None, ['Contract', 'InternetService', 'PaymentMethod'])]}, 
    params={'pairs': [('Contract', 'InternetService', 'PaymentMethod')]}
)
e_skf5.set_node(
    'std', grp='pre', processor=StandardScaler, edges = {'X': [('si', None)]}, params = {}
)
e_skf5.set_node(
    'std_expr1', grp='pre', processor=StandardScaler, edges = {'X': [('expr1', None)]}, params = {}
)
e_skf5.set_node(
    'rb', grp='pre', processor=RobustScaler, edges = {'X': [('si', None)]}, params = {}
)
e_skf5.set_node(
    'rb_expr1', grp='pre', processor=RobustScaler, edges = {'X': [('expr1', None)]}, params = {}
)
e_skf5.build()

Building 12 node(s)
Build 5/5 (100%)                       
Build complete: 12 node(s)


In [29]:
e_skf5.set_node(
    'exprd', grp='pre', processor = ExprProcessor, edges = {'X': [('si', None)]},
    params={'dict_expr': {
        'TotalCharges_3': ((pl.col('si__TotalCharges') * 1e-3).cast(pl.Int8) % 10).cast(pl.String).cast(pl.Categorical)
    }, 'with_columns': False}
)
e_skf5.build()

Affected 1 dependent node(s): ['nn_0']
Building 1 node(s)

Build 5/5 (100%)                  
Build complete: 1 node(s)


In [33]:
e_skf5.set_node(
    'nn_0', grp='nn', 
    edges={'X': [(None, X_bin + X_nom + X_bin2), ('std', None), ('std_expr1', None), ('coov', None), ('com3', None), ('exprd', None)]},
    params={'hidden': {'units': [256, 128], 'dropout': 0.5}, 'cat_cols': ColSelector(col_type = 'category'), 'epochs': 5, 'learning_rate': 0.0002}
)
e_skf5.exp()
e_skf5.get_collector('AUC').get_metrics_agg()[0]

Experimenting 1 node(s)
Exp 5/5 (100%)                                                      
Experimentation complete: 1 node(s)


,valid,train_sub
nn_0,0.915887,0.926159


In [34]:
e_skf5.set_node(
    'nn_1', grp='nn', 
    edges={'X': [(None, X_bin + X_nom + X_bin2), ('std', None), ('std_expr1', None), ('coov', None), ('com3', None), ('exprd', None)]},
    params={'hidden': {'units': [256, 128], 'dropout': 0.5}, 'cat_cols': ColSelector(col_type = 'category'), 'epochs': 5, 'learning_rate': 0.0001}
)
e_skf5.exp()
e_skf5.get_collector('AUC').get_metrics_agg('nn_1')[0]

Experimenting 1 node(s)
Exp 5/5 (100%)                                                      
Experimentation complete: 1 node(s)


,valid,train_sub
nn_1,0.917288,0.923372


In [37]:
e_skf5.set_node(
    'nn_2', grp='nn', 
    edges={'X': [(None, X_bin + X_nom + X_bin2), ('std', None), ('std_expr1', None), ('coov', None), ('com3', None), ('exprd', None)]},
    params={'hidden': {'units': [256, 128], 'dropout': 0.5}, 'cat_cols': ColSelector(col_type = 'category'), 'epochs': 4, 'learning_rate': 0.0001}
)
e_skf5.exp()
e_skf5.get_collector('AUC').get_metrics_agg('nn_2')[0]

Experimenting 1 node(s)
Exp 5/5 (100%)                                                      
Experimentation complete: 1 node(s)


,valid,train_sub
nn_2,0.917427,0.921881
